In [ ]:
from pathlib import Path

import jax
import xarray as xr
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperFluxFNOLocal,
)
from context_flux_no.utils import num_parameters


jax.config.update("jax_default_device", jax.devices("gpu")[2])

In [2]:
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_default_device", jax.devices("gpu")[3])
datadir = Path("../../data")
# dataset_xr = xr.load_dataset(
#     datadir / "cubic_no_source_large_train.hdf5", engine="h5netcdf"
# )
dataset_xr = xr.open_mfdataset(
    sorted(list(datadir.glob("cubic_no_source_large_train_seed=*.hdf5"))),
    combine="nested",
    concat_dim="pde",
    engine="h5netcdf",
).isel(t=slice(None, None, 10))
dataset_xr

OSError: no files to open

In [3]:
hyperfluxfno = HyperFluxFNOLocal(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    depth=4,
    lift_dim=128,
    stencil_size=(10, 10),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno.num_parameters() / 1e6)

/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/encoders/utils.py:50: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Output vector


2.67802


In [4]:
num_parameters(hyperfluxfno.context_encoder)

279424

In [62]:
hyperfluxfno2 = HyperFluxFNOLocal(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="ViT",
    encoder_kwargs=dict(
        patch_size=(1, 4),
        hidden_dim=32,
        num_heads=4,
        num_layers=2,
        dropout=0.0,
        num_hidden_patch=32,
        activation=jax.nn.gelu,
    ),
    depth=4,
    lift_dim=128,
    stencil_size=(10, 10),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno2.num_parameters() / 1e6)

3.077188


In [63]:
num_parameters(hyperfluxfno2.context_encoder)

678592

In [64]:
num_parameters(hyperfluxfno2.context_encoder.patch_embedding)

529536